# Track 1: YOLO for PCB Defect Detection
## Dual-Track Architecture -- Lightweight Edge CNN Branch

**Project:** FOD Predictive Detection / PCB Defect Detection  
**Dataset:** DsPCBSD+ (9 defect classes, 10,259 images at 640x640)  
**Model:** YOLO26n / YOLOv8n (~2.7M parameters)  
**Track 1 Role:** Extreme speed and real-time edge execution  

---

### Hardware Profile & Execution Guidelines
- **GPU (if CUDA enabled):** NVIDIA GeForce RTX 2050 (4 GB VRAM) -> ~25-35 seconds/epoch (~45-60 mins for 100 epochs)
- **CPU Fallback:** AMD Ryzen 5 5500H -> ~20-30 mins/epoch (~35-50 hours for 100 epochs)
- **Batch Size:** Set to 16 for GPU, or automatically scaled down to 8 if running on CPU
- **Local Dataset:** Pre-configured to point directly to `Dataset/DsPCBSD+/Data_YOLO` via `pcb_data.yaml`


## 1. Environment Verification & Hardware Detection


In [ ]:
import os
import sys
import torch
import ultralytics

print("=" * 60)
print("  Environment & Hardware Verification")
print("=" * 60)
print(f"Python Version:   {sys.version.split()[0]}")
print(f"PyTorch Version:  {torch.__version__}")
print(f"Ultralytics:      {ultralytics.__version__}")

CUDA_AVAILABLE = torch.cuda.is_available()
if CUDA_AVAILABLE:
    DEVICE = 0
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"CUDA Available:   YES")
    print(f"Active GPU:       {gpu_name} ({gpu_mem:.1f} GB VRAM)")
    RECOMMENDED_BATCH = 16
else:
    DEVICE = "cpu"
    print(f"CUDA Available:   NO (Running on CPU)")
    print(f"Processor:        {os.environ.get('PROCESSOR_IDENTIFIER', 'Multi-Core CPU')}")
    print("NOTE: Training on CPU is very slow. For fast training, install PyTorch with CUDA or use Google Colab.")
    RECOMMENDED_BATCH = 8

print(f"Selected Device:  {DEVICE}")
print(f"Default Batch:    {RECOMMENDED_BATCH}")
print("=" * 60)


## 2. Dataset Configuration & Verification
Points directly to our local `pcb_data.yaml` and verifies that all 8,208 training images and 2,051 validation images are present.


In [ ]:
import os
import sys
import yaml
import shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    MY_DRIVE = Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive').exists() else Path('/content/drive/My Drive')
    DRIVE_FOLDER_NAME = "Classified_Dataset"
    source_path = MY_DRIVE / DRIVE_FOLDER_NAME
    if not source_path.exists():
        sub = MY_DRIVE / "DataSet_CourseProject" / DRIVE_FOLDER_NAME
        if sub.exists():
            source_path = sub
            
    # Cache / flatten to local NVMe SSD for fast training
    dataset_root = Path("/content/dataset")
    if not dataset_root.exists() or len(list((dataset_root / "images" / "train").glob("*.jpg"))) < 1000:
        print(f"Preparing dataset on fast local drive: {dataset_root} ...")
        for split in ['train', 'val']:
            (dataset_root / 'images' / split).mkdir(parents=True, exist_ok=True)
            (dataset_root / 'labels' / split).mkdir(parents=True, exist_ok=True)
            for f in (source_path / 'images' / split).rglob("*.jpg"):
                shutil.copy2(str(f), str(dataset_root / 'images' / split / f.name))
            for f in (source_path / 'labels' / split).rglob("*.txt"):
                shutil.copy2(str(f), str(dataset_root / 'labels' / split / f.name))
                
    YAML_PATH = dataset_root / "pcb_data.yaml"
    data_cfg = {
        "path": dataset_root.as_posix(),
        "train": "images/train",
        "val": "images/val",
        "nc": 9,
        "names": {0: "SH", 1: "SP", 2: "SC", 3: "OP", 4: "MB", 5: "HB", 6: "CS", 7: "CFO", 8: "BMFO"}
    }
    with open(YAML_PATH, "w") as f:
        yaml.dump(data_cfg, f)
else:
    BASE_DIR = Path.cwd()
    YAML_PATH = BASE_DIR / "pcb_data.yaml"
    if not YAML_PATH.exists():
        YAML_PATH = BASE_DIR / "Dataset" / "DsPCBSD+" / "data.yaml"
    with open(YAML_PATH, "r") as f:
        data_cfg = yaml.safe_load(f)
    dataset_root = Path(data_cfg["path"])

train_img_dir = dataset_root / data_cfg["train"]
val_img_dir = dataset_root / data_cfg["val"]

train_count = len(list(train_img_dir.glob("*.jpg")))
val_count = len(list(val_img_dir.glob("*.jpg")))

print("=" * 60)
print(f"Dataset Path:      {dataset_root}")
print(f"Classes ({data_cfg['nc']}):        {list(data_cfg['names'].values())}")
print(f"Train Images:      {train_count} files")
print(f"Validation Images: {val_count} files")
print("=" * 60)

assert train_count > 0, "No training images found! Please check dataset path."
assert val_count > 0, "No validation images found! Please check dataset path."
print("Dataset verification PASSED successfully.")


## 3. Dataset Visualization -- Ground Truth Samples


In [ ]:
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

CLASS_NAMES = data_cfg["names"]
sample_images = list(train_img_dir.glob("*.jpg"))[:4]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for idx, img_path in enumerate(sample_images):
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    ax = axes[idx]
    ax.imshow(img_rgb)
    ax.set_title(img_path.name, fontsize=10)
    ax.axis("off")
    
    # Load corresponding label file
    lbl_path = dataset_root / "labels" / "train" / (img_path.stem + ".txt")
    if lbl_path.exists():
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    xc, yc, bw, bh = map(float, parts[1:5])
                    x1 = (xc - bw / 2) * w
                    y1 = (yc - bh / 2) * h
                    box_w = bw * w
                    box_h = bh * h
                    
                    rect = patches.Rectangle((x1, y1), box_w, box_h, linewidth=2, edgecolor='red', facecolor='none')
                    ax.add_patch(rect)
                    ax.text(x1, max(y1 - 4, 0), CLASS_NAMES.get(cls_id, str(cls_id)),
                            color='yellow', fontsize=8, fontweight='bold',
                            bbox=dict(facecolor='black', alpha=0.6, pad=1))

plt.suptitle("Sample Ground-Truth PCB Defect Annotations", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Track 1 Model Initialization (YOLO26n / YOLOv8n)


In [ ]:
from ultralytics import YOLO

# Load lightweight nano baseline model
MODEL_WEIGHTS = "yolo26n.pt"
print(f"Loading model: {MODEL_WEIGHTS} ...")
model = YOLO(MODEL_WEIGHTS)

# Display model overview
print(f"Loaded architecture: {model.model.__class__.__name__}")
total_params = sum(p.numel() for p in model.model.parameters())
print(f"Total Parameters: {total_params:,} (~{total_params/1e6:.2f}M)")


## 5. Local Training Execution
Configured with local parameters tailored for consumer hardware:
- `workers=2` (prevents Windows shared memory crashes)
- `batch=RECOMMENDED_BATCH` (16 for GPU, 8 for CPU)
- `amp=True` (Automatic mixed precision for GPU acceleration)
- `patience=20` (Early stopping when convergence is reached)


In [ ]:
# Configure training parameters
EPOCHS = 100       # Set to 5-10 for quick smoke test, or 100 for full convergence
IMG_SIZE = 640     # Standard PCB resolution
PROJECT_DIR = str(BASE_DIR / "runs" / "detect")
RUN_NAME = "PCB_YOLO26n_Local"

print("=" * 60)
print("  Starting Local YOLO Training")
print("=" * 60)
print(f"  Data Config:    {YAML_PATH}")
print(f"  Device:         {DEVICE}")
print(f"  Batch Size:     {RECOMMENDED_BATCH}")
print(f"  Image Size:     {IMG_SIZE}")
print(f"  Epochs:         {EPOCHS}")
print(f"  Workers:        2 (Windows safe)")
print(f"  Output Dir:     {PROJECT_DIR}/{RUN_NAME}")
print("=" * 60)

# Start training
results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=RECOMMENDED_BATCH,
    device=DEVICE,
    workers=2,
    project=PROJECT_DIR,
    name=RUN_NAME,
    patience=20,
    save=True,
    plots=True,
    amp=(DEVICE != "cpu"),
    exist_ok=True
)

BEST_WEIGHTS = Path(results.save_dir) / "weights" / "best.pt"
print(f"\nTraining completed! Best checkpoint saved to: {BEST_WEIGHTS}")


## 6. Model Validation & Per-Class Metrics Evaluation


In [ ]:
import pandas as pd
from ultralytics import YOLO

# Load best checkpoint from training run
checkpoint_to_eval = BEST_WEIGHTS if 'BEST_WEIGHTS' in locals() and BEST_WEIGHTS.exists() else "yolo26n.pt"
eval_model = YOLO(str(checkpoint_to_eval))

print(f"Evaluating model checkpoint: {checkpoint_to_eval}")
val_results = eval_model.val(
    data=str(YAML_PATH),
    split="val",
    imgsz=IMG_SIZE,
    batch=RECOMMENDED_BATCH,
    device=DEVICE
)

# Extract overall metrics
print("\n" + "=" * 60)
print("  Overall Validation Performance")
print("=" * 60)
print(f"  mAP@50:       {val_results.box.map50:.4f}")
print(f"  mAP@50-95:    {val_results.box.map:.4f}")
print(f"  Precision:     {val_results.box.mp:.4f}")
print(f"  Recall:        {val_results.box.mr:.4f}")
print("=" * 60)

# Build per-class DataFrame
names = eval_model.names
class_metrics = []
for i in range(len(names)):
    ap50 = val_results.box.ap50[i]
    ap = val_results.box.ap[i]
    class_metrics.append({
        "Class ID": i,
        "Class Name": names[i],
        "mAP@50": round(float(ap50), 4),
        "mAP@50-95": round(float(ap), 4)
    })

df_metrics = pd.DataFrame(class_metrics)
print("\nPer-Class Detection Accuracy:")
print(df_metrics.to_string(index=False))


## 7. Qualitative Predictions on Test/Validation Images


In [ ]:
import random
from PIL import Image

sample_val_imgs = list(val_img_dir.glob("*.jpg"))
chosen_samples = random.sample(sample_val_imgs, min(4, len(sample_val_imgs)))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for idx, img_p in enumerate(chosen_samples):
    pred = eval_model.predict(source=str(img_p), conf=0.25, imgsz=IMG_SIZE, device=DEVICE, verbose=False)[0]
    plotted = pred.plot() # Returns BGR numpy array
    plotted_rgb = cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB)
    
    ax = axes[idx]
    ax.imshow(plotted_rgb)
    ax.set_title(f"{img_p.name}\n({len(pred.boxes)} detections)", fontsize=10)
    ax.axis("off")

plt.suptitle("YOLO Track 1 Predictions (Confidence >= 0.25)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Model Export for Edge Deployment (ONNX / FP16)
Exports the model into an open interoperable format for edge inference (NVIDIA Jetson, Raspberry Pi, ONNX Runtime).


In [ ]:
print("Exporting trained YOLO model to ONNX format...")
try:
    onnx_path = eval_model.export(format="onnx", imgsz=IMG_SIZE, half=(DEVICE != "cpu"), simplify=True)
    print(f"Export successful! ONNX model saved at: {onnx_path}")
    
    size_pt = os.path.getsize(str(checkpoint_to_eval)) / (1024 * 1024)
    size_onnx = os.path.getsize(str(onnx_path)) / (1024 * 1024)
    print(f"PyTorch size:  {size_pt:.2f} MB")
    print(f"ONNX size:     {size_onnx:.2f} MB")
except Exception as e:
    print(f"Export warning: {e}")
